# 06 pymefpp SWarp

This notebooks run SWarp over the images for which Scamp creates a .head file. <br>
Tested with SWarp 2.38.0.

Uses the default configuration for SWarp, in case of needs, additional configuration parameters can be changed to optimize the performance in Chapman for example

In [ ]:
import os
import glob
import logging
from astropy.io import fits
from mefpp_astromatic import SWarp

In [ ]:
def select_observations(image, observation, filt):
    """Determine if image name contains the observation
    and the filter string in the name. Return boolean."""
    
    if observation in image and filt in image:
        return True
    else:
        return False
    
    
def run_swarp(swarp_dir, frame_list):
    swarp = SWarp(swarp_dir)
    
    # creates the swap and resamp folders
    if not os.path.isdir("./resamp"):
        os.mkdir("./resamp")
    if not os.path.isdir("./swap"):
        os.mkdir("./swap")
    
    # updates some SWarp parameters
    swarp.config["IMAGEOUT_NAME"] = frame_list[:-5] + ".coadd.fits"
    swarp.config["WEIGHTOUT_NAME"] = frame_list[:-5] + ".weight.fits"
    swarp.config["RESAMPLE_DIR"] = "./resamp"
    swarp.config["VMEM_DIR"] = "./swap"
    swarp.config["COPY_KEYWORDS"] = "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"
    
    if "NB_659" in frame_list:
        swarp.config["SUBTRACT_BACK"] = "N"
    
    logging.info("Running Swarp on framelist")
    swarp.run('@'+ frame_list, updateconfig=True, clean=False)
    
    
def update_headers(swarped_image, reference_image):
    """
    Read the headers of the reference image to transfer keywords OBJECT, RA and DEC 
    which were not copied during SWarp
    """

    logging.info("reference_image: {}".format(reference_image))
    reference_header = fits.getheader(reference_image)
    with fits.open(swarped_image, mode="update", output_verify="ignore") as hdul:
        hdr = hdul[0].header
        hdr["BUNIT"] = ("e-/s", "units of image pixels")
        hdr["ORIGIN"] = reference_header["ORIGIN"]
        hdr["OBJECT"] = reference_header["OBJECT"]
        hdr["RA"] = hdr["CRVAL1"]
        hdr["DEC"] = hdr["CRVAL2"]    


def prepare_swarp(science_dir, images_list):
    """
    Run swarp on the output of scamp for a given observation and filter.
    Creates the Swarp directories where the swarped image and the logfiles
    will be stored (unique to observation).
    """
    observation = images_list[0].split("/")[-5]
    filter_ = images_list[0].split("/")[-4]
    logging.info("filter_: {}".format(filter_))
    swarp_dir = science_dir + observation + "/Swarp_dir/"
    logging.info("\n          - observation: {0}".format(observation))
    logging.info("\n          - filter_: {0}".format(filter_))
    logging.info("\n          - swarp_dir: {0}".format(swarp_dir))
    logging.info("\n          - image_list[0]: {0}".format(images_list[0]))

    logging.info("Attempting to create the SWarp folder..")
    if not os.path.isdir(swarp_dir):
        os.mkdir(swarp_dir)
        logging.info("Folder created")
    else:
        logging.info("Folder already exists")
        
    current_dir = os.getcwd()
    os.chdir(swarp_dir)
    
    # we create the list with images to swarp
    with open(observation + "_" + filter_ + ".list", "w") as f:
        for image in images_list:
            f.write(image[:-5]+".fits\n")
    
    # now we can run swarp
    run_swarp(swarp_dir, observation + "_" + filter_ + ".list")
    
    # update the headers of the swarped image
    swarp_observation = swarp_dir+'/'+observation
    swarp_filter = filter_
    logging.info("swarp_observation: {0}  swarp_filter: {1}".format(swarp_observation, swarp_filter))
    swarp_coadded_image = swarp_observation + "_" + swarp_filter + ".coadd.fits"
    header_reference_image = images_list[0][:-5]+'.fits'
    logging.info("\nUpdating headers of image: {0}\nwith headers from {1}".format(swarp_coadded_image, header_reference_image))
    update_headers(swarp_coadded_image, header_reference_image)
    
    # back to the original folder
    os.chdir(current_dir)

In [ ]:
#root_dir = "/Users/fselman/Pictures/OCAM/CPPup/reduced/"  # the root directory where the calibrated and science images will go
root_dir = "/Users/fselman/Pictures/OCAM/M67/"
reduced_science_dir = root_dir+'science/'
log_dir = root_dir+'logs/'

log_file_swarp = log_dir + "/pymeffpp_swarp.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# all this part is to select all the .head files and grouping accordling to observation and filter
# the grouped images are stored in the swarp_list dictionary, where the key correspond to the combination
# of observation plus filter, and the element to corresponding list of images
# print(glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.head"))
observations = []
filters = []
targets_to_swarp = []
for target in glob.glob(reduced_science_dir+"/*/*/*/*_pre/*_wcs.head"):
    targets_to_swarp.append(target)
    observations.append(target.split("/")[-5])
    filters.append(target.split("/")[-4])
    
observations = set(observations)
filters = set(filters)

swarp_lists = {}
for observation in observations:
    for filter_ in filters:
        list_to_swarp = []
        filtered_images = filter(lambda x: select_observations(x, observation, filter_), targets_to_swarp)
        #logging.info(observation + "/" + filter_)
        #for animage in list(filtered_images):
        #    logging.info("     --> {}".format(animage))
        swarp_lists[observation + "/" + filter_] = list(filtered_images)
             
for observation_tag in swarp_lists.keys():
    logging.info("SWarping observation {}, filter {}".format(observation_tag.split("/")[0],
                                                             observation_tag.split("/")[1]))
    prepare_swarp(reduced_science_dir, swarp_lists[observation_tag])
    
    
logging.info('-------------- DONE ---------------') 